# Phase 2 — RPi 용도 추천 AI: SFT (Qwen2.5-1.5B-Instruct)

**런타임 설정**: 상단 메뉴 `런타임(Runtime) > 런타임 유형 변경 > 하드웨어 가속기: T4 GPU` 선택 후 실행.

이 노트북은 [wqrvQ2WR/rpi-ai](https://github.com/wqrvQ2WR/rpi-ai) 저장소의 Phase 1 데이터셋(`data/sft_train.jsonl`, `data/sft_val.jsonl`)을 사용해 `Qwen/Qwen2.5-1.5B-Instruct`를 LoRA로 SFT 합니다.

- GPU: T4 (무료 티어) — bf16 텐서코어가 없으므로 **fp16**을 사용합니다.
- 방법: PEFT LoRA (전체 파인튜닝은 옵티마이저 상태 때문에 T4 16GB에 부적합)
- 라이브러리: `transformers`, `trl`, `peft`, `accelerate`, `datasets`


In [ ]:
!nvidia-smi

In [ ]:
!pip uninstall -y -q torchao
!pip install -q -U "transformers>=4.46" "trl>=0.12" peft accelerate datasets

## 1. 데이터 다운로드 (GitHub raw, 공개 저장소라 인증 불필요)

In [ ]:
import urllib.request, os

os.makedirs("data", exist_ok=True)
BASE = "https://raw.githubusercontent.com/wqrvQ2WR/rpi-ai/main/data"
for fname in ["sft_train.jsonl", "sft_val.jsonl"]:
    urllib.request.urlretrieve(f"{BASE}/{fname}", f"data/{fname}")
    print(fname, os.path.getsize(f"data/{fname}"), "bytes")


## 2. 데이터셋 로드

In [ ]:
from datasets import load_dataset

train_ds = load_dataset("json", data_files="data/sft_train.jsonl", split="train")
val_ds = load_dataset("json", data_files="data/sft_val.jsonl", split="train")
print(train_ds)
print(train_ds[0])


## 3. 모델 / 토크나이저 로드

`Qwen/Qwen2.5-1.5B-Instruct`는 이미 채팅 템플릿이 내장되어 있어서 `messages` 컬럼을 그대로
SFTTrainer에 넘기면 자동으로 템플릿이 적용됩니다.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)


## 4. LoRA 설정

In [ ]:
from peft import LoraConfig

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)


## 5. 학습

In [ ]:
from trl import SFTConfig, SFTTrainer

sft_config = SFTConfig(
    output_dir="rpi-ai-qwen2.5-1.5b-sft",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=5,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    max_length=1024,
    fp16=True,
    bf16=False,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    peft_config=peft_config,
    processing_class=tokenizer,
)

trainer.train()

## 6. LoRA 어댑터 저장 + 베이스 모델과 병합

In [ ]:
trainer.save_model("rpi-ai-qwen2.5-1.5b-sft/adapter")
tokenizer.save_pretrained("rpi-ai-qwen2.5-1.5b-sft/adapter")

from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map="auto")
merged = PeftModel.from_pretrained(base, "rpi-ai-qwen2.5-1.5b-sft/adapter")
merged = merged.merge_and_unload()
merged.save_pretrained("rpi-ai-qwen2.5-1.5b-sft/merged")
tokenizer.save_pretrained("rpi-ai-qwen2.5-1.5b-sft/merged")
print("merged 모델 저장 완료: rpi-ai-qwen2.5-1.5b-sft/merged")


## 7. 빠른 확인

In [ ]:
from transformers import pipeline

pipe = pipeline("text-generation", model=merged, tokenizer=tokenizer, max_new_tokens=200)
messages = [
    {"role": "system", "content": "당신은 라즈베리파이(Raspberry Pi) 프로젝트와 용도를 추천해주는 AI 어시스턴트입니다. 사용자의 상황과 목적에 맞는 프로젝트, 필요한 하드웨어, 난이도를 함께 안내하세요."},
    {"role": "user", "content": "집에서 광고 없이 인터넷 쓰고 싶어"},
]
out = pipe(messages)
print(out[0]["generated_text"][-1]["content"])


## 8. 결과 내려받기 (Google Drive에 저장하거나 zip으로 다운로드)

Colab 세션이 끝나면 로컬 파일이 사라지므로 `merged/` 폴더를 Drive에 복사하거나 zip으로
다운로드해두세요. GGUF 변환(Phase 5)은 별도 llama.cpp 단계에서 이 `merged/` 폴더를 입력으로
사용합니다.

In [ ]:
# 방법 A: Google Drive에 저장
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# shutil.copytree("rpi-ai-qwen2.5-1.5b-sft/merged", "/content/drive/MyDrive/rpi-ai-merged", dirs_exist_ok=True)

# 방법 B: zip으로 다운로드
import shutil
shutil.make_archive("rpi-ai-merged", "zip", "rpi-ai-qwen2.5-1.5b-sft/merged")

from google.colab import files
files.download("rpi-ai-merged.zip")
